In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print('Sem erro aqui')

Sem erro aqui


In [4]:
df = pd.read_csv('heart_disease_dataset.csv')
print(f'{df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()

1000 linhas x 16 colunas


,Age,Gender,Cholesterol,Blood Pressure,Heart Rate,Smoking,Alcohol Intake,Exercise Hours,Family History,Diabetes,Obesity,Stress Level,Blood Sugar,Exercise Induced Angina,Chest Pain Type,Heart Disease
0,75,Female,228,119,66,Current,Heavy,1,No,No,Yes,8,119,Yes,Atypical Angina,1
1,48,Male,204,165,62,Current,NaN,5,No,No,No,9,70,Yes,Typical Angina,0
2,53,Male,234,91,67,Never,Heavy,3,Yes,No,Yes,5,196,Yes,Atypical Angina,1
3,69,Female,192,90,72,Current,NaN,4,No,Yes,No,7,107,Yes,Non-anginal Pain,0
4,62,Female,172,163,93,Never,NaN,6,No,Yes,No,2,183,Yes,Asymptomatic,0


In [10]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

target = 'Heart Disease'
X = df.drop(columns=[target])
y = df[target]

if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y.astype(str))

print('Transformacoes concluidas.')
print(f'X: {X.shape} | y: {y.shape}')

Transformacoes concluidas.
X: (1000, 15) | y: (1000,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]}')

Treino: 800 | Teste: 200


In [12]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print('KNN e Regressao Logistica')

KNN e Regressao Logistica


In [20]:
def metricas(y_real, y_pred):
    return {
        'Acuracia': accuracy_score(y_real, y_pred),
        'Precisao': precision_score(y_real, y_pred, zero_division=0),
        'Recall': recall_score(y_real, y_pred, zero_division=0),
        'F1': f1_score(y_real, y_pred, zero_division=0)
    }

m_knn = metricas(y_test, y_pred_knn)
m_lr = metricas(y_test, y_pred_lr)

comparacao = pd.DataFrame({'KNN': m_knn, 'Regressao Logistica': m_lr}).round(4)
print(comparacao)

melhor = 'KNN' if m_knn['F1'] >= m_lr['F1'] else 'Regressao Logistica'
print(f'Melhor algoritmo (criterio F1): {melhor}')

             KNN  Regressao Logistica
Acuracia  0.8350               0.8650
Precisao  0.7848               0.8493
Recall    0.7949               0.7949
F1        0.7898               0.8212
Melhor algoritmo (criterio F1): Regressao Logistica


In [21]:
baseline_knn = m_knn['F1']
baseline_lr = m_lr['F1']

resultados = []
for feat in X.columns:
    cols = [c for c in X.columns if c != feat]

    Xtr = X_train[cols]
    Xte = X_test[cols]

    sc = StandardScaler()
    Xtr_sc = sc.fit_transform(Xtr)
    Xte_sc = sc.transform(Xte)

    knn_tmp = KNeighborsClassifier(n_neighbors=5)
    knn_tmp.fit(Xtr_sc, y_train)
    f1_knn_sem = f1_score(y_test, knn_tmp.predict(Xte_sc), zero_division=0)

    lr_tmp = LogisticRegression(max_iter=1000, random_state=42)
    lr_tmp.fit(Xtr_sc, y_train)
    f1_lr_sem = f1_score(y_test, lr_tmp.predict(Xte_sc), zero_division=0)

    impacto_knn = baseline_knn - f1_knn_sem
    impacto_lr = baseline_lr - f1_lr_sem

    resultados.append({
        'Feature': feat,
        'Impacto_F1_KNN': impacto_knn,
        'Impacto_F1_LR': impacto_lr,
        'Impacto_F1_Medio': (impacto_knn + impacto_lr) / 2
    })

impacto_df = pd.DataFrame(resultados).sort_values('Impacto_F1_Medio', ascending=True)

print('Features mais dificeis de aprender (top 5):')
print(impacto_df.head(5).round(4))

Features mais dificeis de aprender (top 5):
                    Feature  Impacto_F1_KNN  Impacto_F1_LR  Impacto_F1_Medio
9                  Diabetes         -0.0867        -0.0087           -0.0477
4                Heart Rate         -0.0404        -0.0078           -0.0241
13  Exercise Induced Angina         -0.0477         0.0000           -0.0238
8            Family History         -0.0154        -0.0023           -0.0089
5                   Smoking         -0.0154         0.0000           -0.0077


### Interpretacao
- Quanto menor (ou negativo) o impacto ao remover uma feature, mais dificil foi para os algoritmos aprenderem padroes uteis dela.
- A coluna Impacto_F1_Medio resume o comportamento conjunto de KNN e Regressao Logistica.